# EAG-UNet: comprehensive, leakage-audited multi-seed experiment report
One executable Jupyter notebook; 15 trained runs (five models x three seeds), including per-image validation, statistical comparisons, transfer metrics, qualitative segmentation, and real segmentation-targeted Grad-CAM.

IMPORTANT: This notebook does not call model.fit(), change weights or perform training. Results are not fabricated.
FIVES official test: 200 annotation records corresponding to 199 unique image-byte hashes. DRIVE transfer: 20 images from the official DRIVE TRAINING partition (not official test). Training and validation grouping is based on exact SHA-256 image bytes, not independently verified patient IDs.
EAG-UNet uses the conventional additive attention gate, and does not demonstrate new operator novelty. Old ablation/VDRG outputs are from a different evaluation protocol and are deliberately excluded.

In [ ]:
from pathlib import Path
import os,json,hashlib,gc
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy import stats
from PIL import Image
EXP=Path(os.environ.get('EAG_EXPERIMENT_ROOT',str(Path.cwd().parent))).resolve()
ROOT=EXP.parent
RUNS=EXP/'LEAKSAFE_FIVES_RETRAIN_V1'/'full'
SPLITS=EXP/'LEAKSAFE_FIVES_SPLITS_V1'
OUT=EXP/'EAG_UNET_GITHUB_PACKAGE';FIG=OUT/'figures';TAB=OUT/'results'
FIG.mkdir(parents=True,exist_ok=True);TAB.mkdir(parents=True,exist_ok=True)
MODELS=['U_Net','ResUNet','Attention_UNet','EAG_UNet','MS_SE_Attention_UNet']
NAMES={'U_Net':'U-Net','ResUNet':'ResUNet','Attention_UNet':'Attention U-Net',
       'EAG_UNet':'EAG-UNet','MS_SE_Attention_UNet':'MS-SE Attention U-Net'}
SEEDS=[42,123,2025]
RUN_GRADCAM=True    # False = only cached CSV summaries, no TensorFlow inference.
CASE_QUANTILES=[.10,.50,.90]
GRADCAM_SEED=42
assert RUNS.is_dir() and (SPLITS/'audit_manifest.json').is_file()
print('Evidence:',RUNS)

## 1. Independent audit of all saved checkpoints and per-image scores

In [ ]:
audit=json.loads((SPLITS/'audit_manifest.json').read_text(encoding='utf8'))
assert audit['sha256_grouped'] and audit['official_test_unique_images']==199
assert audit['clinical_patient_grouping']=='unverified'
records=[];image_parts=[]
for m in MODELS:
    for seed in SEEDS:
        p=RUNS/m/f'seed_{seed}'
        required=['evaluation_summary.json','train_summary.json','best.weights.h5',
                  'test_metrics.csv','threshold_validation_curve.csv','DRIVE_train_1st_manual_external_metrics.csv']
        assert all((p/n).is_file() for n in required),(m,seed)
        summary=json.loads((p/'evaluation_summary.json').read_text(encoding='utf8'))
        train=json.loads((p/'train_summary.json').read_text(encoding='utf8'))
        imgs=pd.read_csv(p/'test_metrics.csv')
        transfer=pd.read_csv(p/'DRIVE_train_1st_manual_external_metrics.csv')
        assert len(imgs)==200 and imgs.image_sha256.nunique()==199 and len(transfer)==20
        calc=imgs.groupby('image_sha256').Dice.mean().mean()
        assert np.isclose(calc,summary['Dice_unique_mean'],rtol=0,atol=1e-12)
        assert train['split_type']=='SHA256-group-disjoint'
        assert train['n_train']==510 and train['n_validation']==90
        assert (p/'best.weights.h5').stat().st_size>100000
        records.append(dict(**summary,parameters=train['parameters'],
                            epochs=train['n_epochs_logged'],checkpoint_bytes=(p/'best.weights.h5').stat().st_size))
        image_parts.append(imgs)
raw=pd.DataFrame(records)
per_image=pd.concat(image_parts,ignore_index=True)
assert len(raw)==15 and raw.groupby('model').seed.nunique().eq(3).all()
raw.to_csv(TAB/'run_level_metrics_15.csv',index=False)
per_image.drop(columns=['input_path','reference_mask'],errors='ignore').to_csv(TAB/'per_image_metrics_15.csv',index=False)
print('PASS 15/15 completed; checkpoint files, per-image Dice and external transfer counts independently checked')

## 2. Three-seed mean/SD/Student-t confidence intervals and paired hypothesis tests

In [ ]:
columns=['Dice_unique_mean','IoU_unique_mean','clDice_unique_mean',
         'Boundary_F1_tol2_unique_mean','Thin_centerline_recall_tol2_unique_mean',
         'Precision_unique_mean','Sensitivity_unique_mean','DRIVE_Dice_mean']
rows=[]
for m in MODELS:
    q=raw[raw.model==m]
    z={'model':m,'name':NAMES[m],'n_seeds':3,'parameters':int(q.parameters.iloc[0])}
    for c in columns:
        x=q[c].to_numpy(float)
        z[c+'_mean']=x.mean();z[c+'_sd']=x.std(ddof=1)
        z[c+'_ci95_half']=stats.t.ppf(.975,2)*stats.sem(x)
    rows.append(z)
summary=pd.DataFrame(rows)
summary.to_csv(TAB/'model_metrics_mean_sd_ci95.csv',index=False)
pivot=raw.pivot(index='seed',columns='model',values='Dice_unique_mean')
comparisons=[('EAG_UNet',n) for n in ['U_Net','ResUNet','Attention_UNet','MS_SE_Attention_UNet']]
comparisons+=[('MS_SE_Attention_UNet','Attention_UNet')]
pairs=[]
for a,b in comparisons:
    d=pivot[a]-pivot[b]
    low,high=stats.t.interval(.95,2,loc=d.mean(),scale=stats.sem(d))
    pairs.append(dict(candidate=a,reference=b,mean_difference=d.mean(),ci95_low=low,
                      ci95_high=high,p_value=stats.ttest_1samp(d,0).pvalue,n=3))
paired=pd.DataFrame(pairs)
paired.to_csv(TAB/'paired_Dice_t_tests.csv',index=False)
print(summary[['name','parameters','Dice_unique_mean_mean','Dice_unique_mean_sd',
               'clDice_unique_mean_mean','DRIVE_Dice_mean_mean']].to_string(index=False))
print(paired.to_string(index=False))

## 3. Figures computed from recorded quantitative evidence

In [ ]:
plt.rcParams.update({'figure.dpi':100,'savefig.dpi':130,'font.size':10,
                     'axes.grid':True,'grid.alpha':.18,'figure.facecolor':'white'})
def save(name):
    plt.savefig(FIG/name,bbox_inches='tight',facecolor='white')
    plt.show();plt.close()
names=[NAMES[m] for m in MODELS]
fig,ax=plt.subplots(figsize=(10,4.8))
v=summary['Dice_unique_mean_mean'];e=summary['Dice_unique_mean_sd']
bars=ax.bar(names,v,yerr=e,capsize=4)
for b,y in zip(bars,v):ax.text(b.get_x()+b.get_width()/2,y+.015,f'{y:.4f}',ha='center',fontsize=9)
ax.set_ylim(0,1);ax.tick_params(axis='x',labelrotation=16)
ax.set_ylabel('Dice, 199 unique FIVES images');ax.set_title('Figure 1. FIVES Dice (3 seeds, mean and SD)')
fig.tight_layout();save('fig01_fives_dice.png')

cols=[('IoU_unique_mean','IoU'),('clDice_unique_mean','clDice'),
      ('Boundary_F1_tol2_unique_mean','Boundary F1 (2 px)'),
      ('Thin_centerline_recall_tol2_unique_mean','Thin-vessel recall')]
fig,axs=plt.subplots(2,2,figsize=(11,7))
for ax,(key,label) in zip(axs.ravel(),cols):
    ax.errorbar(np.arange(5),summary[key+'_mean'],yerr=summary[key+'_sd'],marker='o',capsize=4)
    ax.set_xticks(np.arange(5),['U-Net','ResU','AttU','EAG','MS-SE'])
    ax.set_ylim(0,1);ax.set_ylabel(label)
fig.suptitle('Figure 2. Region, boundary and topology-sensitive metrics')
fig.tight_layout();save('fig02_four_metrics.png')

fig,ax=plt.subplots(figsize=(10,4.8))
ax.bar(names,summary['DRIVE_Dice_mean_mean'],yerr=summary['DRIVE_Dice_mean_sd'],capsize=4)
ax.set_ylim(0,1);ax.tick_params(axis='x',labelrotation=16)
ax.set_title('Figure 3. DRIVE official TRAINING images: transfer, not test benchmark')
ax.set_ylabel('Dice, 20 images / 3 seeds')
fig.tight_layout();save('fig03_drive_training_transfer.png')

fig,ax=plt.subplots(figsize=(10,4.8))
for m in MODELS:
    q=raw[raw.model==m].sort_values('seed')
    ax.plot([str(s) for s in q.seed],q.Dice_unique_mean,marker='o',label=NAMES[m])
ax.set_ylim(.80,.93);ax.set_ylabel('FIVES Dice');ax.set_xlabel('Random seed')
ax.set_title('Figure 4. Across-seed variability');ax.legend(fontsize=8)
fig.tight_layout();save('fig04_seed_variability.png')

## 4. Validation threshold sensitivity and per-image failure distributions

In [ ]:
thresholds=[]
for m in MODELS:
    for s in SEEDS:
        q=pd.read_csv(RUNS/m/f'seed_{s}'/'threshold_validation_curve.csv')
        assert len(q)==17
        q['model']=m;q['seed']=s
        thresholds.append(q)
thresholds=pd.concat(thresholds,ignore_index=True)
thresholds.to_csv(TAB/'validation_threshold_curves.csv',index=False)
tc=next(c for c in thresholds.columns if 'threshold' in c.lower())
dc=next(c for c in thresholds.columns if 'dice' in c.lower())
fig,ax=plt.subplots(figsize=(10,4.8))
for m in MODELS:
    q=thresholds[thresholds.model==m].groupby(tc,as_index=False)[dc].mean()
    ax.plot(q[tc],q[dc],marker='.',label=NAMES[m])
ax.set_xlabel('Threshold selected using validation only');ax.set_ylabel('Validation Dice')
ax.set_title('Figure 5. Threshold sensitivity');ax.legend(fontsize=8)
fig.tight_layout();save('fig05_threshold_sensitivity.png')

fig,ax=plt.subplots(figsize=(10,4.8))
groups=[per_image[(per_image.model==m)&(per_image.seed==42)].groupby('image_sha256').Dice.mean().values for m in MODELS]
ax.boxplot(groups,tick_labels=['U-Net','ResU','AttU','EAG','MS-SE'])
ax.set_ylim(0,1);ax.set_ylabel('Per-image Dice')
ax.set_title('Figure 6. Image-level Dice distribution, seed 42')
fig.tight_layout();save('fig06_image_level_boxplots.png')

eag=per_image[(per_image.model=='EAG_UNet')&(per_image.seed==42)].sort_values(['Dice','image_sha256']).drop_duplicates('image_sha256')
selected=eag.iloc[[int(round((len(eag)-1)*x)) for x in CASE_QUANTILES]].copy()
assert len(selected)==3 and selected.image_sha256.nunique()==3
selected['selection_quantile']=CASE_QUANTILES
selected[['id','image_sha256','Dice','selection_quantile']].to_csv(
    TAB/'qualitative_case_selection.csv',index=False)
print('Predetermined 10th/50th/90th percentile examples, seed 42:')
print(selected[['id','Dice','selection_quantile']].to_string(index=False))

## 5. Same architecture functions as original training (verbatim extracted from experiment)

In [ ]:
import tensorflow as tf, cv2
from tensorflow.keras import layers,models
IMG_SIZE=(512,512)
for device in tf.config.list_physical_devices('GPU'):
    try:tf.config.experimental.set_memory_growth(device,True)
    except RuntimeError:pass
print('TensorFlow',tf.__version__)

In [ ]:
def conv_block(x, filters, name=None):
    p = "" if name is None else name + "_"
    x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal", name=p+"conv1")(x)
    x = layers.BatchNormalization(name=p+"bn1")(x)
    x = layers.Activation("relu", name=p+"relu1")(x)
    x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal", name=p+"conv2")(x)
    x = layers.BatchNormalization(name=p+"bn2")(x)
    x = layers.Activation("relu", name=p+"relu2")(x)
    return x

def residual_block(x, filters, name=None):
    p = "" if name is None else name + "_"
    shortcut = layers.Conv2D(filters, 1, padding="same", kernel_initializer="he_normal", name=p+"shortcut_conv")(x)
    shortcut = layers.BatchNormalization(name=p+"shortcut_bn")(shortcut)
    y = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal", name=p+"conv1")(x)
    y = layers.BatchNormalization(name=p+"bn1")(y)
    y = layers.Activation("relu", name=p+"relu1")(y)
    y = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal", name=p+"conv2")(y)
    y = layers.BatchNormalization(name=p+"bn2")(y)
    y = layers.Add(name=p+"add")([shortcut, y])
    y = layers.Activation("relu", name=p+"relu2")(y)
    return y

def se_block(x, ratio=8, name=None):
    p = "" if name is None else name + "_"
    ch = int(x.shape[-1])
    s = layers.GlobalAveragePooling2D(name=p+"gap")(x)
    s = layers.Dense(max(ch // ratio, 1), activation="relu", name=p+"fc1")(s)
    s = layers.Dense(ch, activation="sigmoid", name=p+"fc2")(s)
    s = layers.Reshape((1, 1, ch), name=p+"reshape")(s)
    return layers.Multiply(name=p+"scale")([x, s])

def ms_se_block(x, filters, name=None):
    p = "" if name is None else name + "_"
    b3 = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal", name=p+"conv3")(x)
    b5 = layers.Conv2D(filters, 5, padding="same", kernel_initializer="he_normal", name=p+"conv5")(x)
    y = layers.Concatenate(name=p+"concat")([b3, b5])
    y = layers.Conv2D(filters, 1, padding="same", kernel_initializer="he_normal", name=p+"project")(y)
    y = layers.BatchNormalization(name=p+"bn")(y)
    y = layers.Activation("relu", name=p+"relu")(y)
    y = se_block(y, name=p+"se")
    return y

def attention_gate(skip, gating, inter_filters=None, name=None):
    p = "" if name is None else name + "_"
    ch = int(skip.shape[-1])
    if inter_filters is None:
        inter_filters = max(ch // 2, 1)
    theta_x = layers.Conv2D(inter_filters, 1, padding="same", name=p+"theta_x")(skip)
    phi_g = layers.Conv2D(inter_filters, 1, padding="same", name=p+"phi_g")(gating)
    add = layers.Add(name=p+"add")([theta_x, phi_g])
    act = layers.Activation("relu", name=p+"relu")(add)
    psi = layers.Conv2D(1, 1, padding="same", name=p+"psi")(act)
    alpha = layers.Activation("sigmoid", name=p+"sigmoid")(psi)
    refined = layers.Multiply(name=p+"multiply")([skip, alpha])
    return refined

def encoder_path(inputs, base=24, block_fn=conv_block):
    e1 = block_fn(inputs, base, "enc1")
    p1 = layers.MaxPooling2D(2, name="pool1")(e1)
    e2 = block_fn(p1, base*2, "enc2")
    p2 = layers.MaxPooling2D(2, name="pool2")(e2)
    e3 = block_fn(p2, base*4, "enc3")
    p3 = layers.MaxPooling2D(2, name="pool3")(e3)
    e4 = block_fn(p3, base*8, "enc4")
    p4 = layers.MaxPooling2D(2, name="pool4")(e4)
    b = block_fn(p4, base*16, "bottleneck")
    return [e1, e2, e3, e4], b

def decoder_step(x, skip, filters, name, use_gate=False, gate=True):
    x = layers.UpSampling2D(size=(2,2), interpolation="bilinear", name=name+"_upsample")(x)
    x = layers.Conv2D(filters, 1, padding="same", kernel_initializer="he_normal", name=name+"_proj")(x)
    if use_gate and gate:
        skip = attention_gate(skip, x, inter_filters=max(filters//2, 1), name=name+"_ag")
    x = layers.Concatenate(name=name+"_concat")([x, skip])
    x = conv_block(x, filters, name=name+"_conv")
    return x

def build_unet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), base=24):
    inputs = layers.Input(input_shape, name="input")
    skips, b = encoder_path(inputs, base=base, block_fn=conv_block)
    x = decoder_step(b, skips[3], base*8, "dec4", use_gate=False)
    x = decoder_step(x, skips[2], base*4, "dec3", use_gate=False)
    x = decoder_step(x, skips[1], base*2, "dec2", use_gate=False)
    x = decoder_step(x, skips[0], base, "dec1", use_gate=False)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="output")(x)
    return models.Model(inputs, out, name=f"U_Net_B{base}")

def build_resunet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), base=24):
    inputs = layers.Input(input_shape, name="input")
    skips, b = encoder_path(inputs, base=base, block_fn=residual_block)
    x = decoder_step(b, skips[3], base*8, "dec4", use_gate=False)
    x = decoder_step(x, skips[2], base*4, "dec3", use_gate=False)
    x = decoder_step(x, skips[1], base*2, "dec2", use_gate=False)
    x = decoder_step(x, skips[0], base, "dec1", use_gate=False)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="output")(x)
    return models.Model(inputs, out, name=f"ResUNet_B{base}")

def build_attention_unet_standard(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), base=24):
    # Standard Attention U-Net baseline: attention gates applied to every skip.
    inputs = layers.Input(input_shape, name="input")
    skips, b = encoder_path(inputs, base=base, block_fn=conv_block)
    x = decoder_step(b, skips[3], base*8, "dec4", use_gate=True, gate=True)
    x = decoder_step(x, skips[2], base*4, "dec3", use_gate=True, gate=True)
    x = decoder_step(x, skips[1], base*2, "dec2", use_gate=True, gate=True)
    x = decoder_step(x, skips[0], base, "dec1", use_gate=True, gate=True)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="output")(x)
    return models.Model(inputs, out, name=f"Attention_UNet_B{base}")

def build_ms_se_attention_unet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), base=24):
    inputs = layers.Input(input_shape, name="input")
    def block(x, f, name):
        return ms_se_block(x, f, name=name)
    skips, b = encoder_path(inputs, base=base, block_fn=block)
    x = decoder_step(b, skips[3], base*8, "dec4", use_gate=True, gate=True)
    x = decoder_step(x, skips[2], base*4, "dec3", use_gate=True, gate=True)
    x = decoder_step(x, skips[1], base*2, "dec2", use_gate=True, gate=True)
    x = decoder_step(x, skips[0], base, "dec1", use_gate=True, gate=True)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="output")(x)
    return models.Model(inputs, out, name=f"MS_SE_Attention_UNet_B{base}")

def build_eag_unet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), base=24, gate_mode="full"):
    # gate_mode: full, none, shallow, deep
    inputs = layers.Input(input_shape, name="input")
    skips, b = encoder_path(inputs, base=base, block_fn=conv_block)
    # dec4 is deepest skip; dec1 is shallowest/highest-resolution skip.
    gate_dec4 = gate_mode in ["full", "deep"]
    gate_dec3 = gate_mode in ["full", "deep"]
    gate_dec2 = gate_mode in ["full", "shallow"]
    gate_dec1 = gate_mode in ["full", "shallow"]
    x = decoder_step(b, skips[3], base*8, "dec4", use_gate=True, gate=gate_dec4)
    x = decoder_step(x, skips[2], base*4, "dec3", use_gate=True, gate=gate_dec3)
    x = decoder_step(x, skips[1], base*2, "dec2", use_gate=True, gate=gate_dec2)
    x = decoder_step(x, skips[0], base, "dec1", use_gate=True, gate=gate_dec1)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="output")(x)
    return models.Model(inputs, out, name=f"EAG_UNet_B{base}_{gate_mode}")

def get_model(model_name):
    if model_name == "U_Net":
        return build_unet(base=24)
    if model_name == "ResUNet":
        return build_resunet(base=24)
    if model_name == "Attention_UNet":
        return build_attention_unet_standard(base=24)
    if model_name == "MS_SE_Attention_UNet":
        return build_ms_se_attention_unet(base=24)
    if model_name == "EAG_UNet":
        return build_eag_unet(base=24, gate_mode="full")
    if model_name == "EAG_NoGate_B24":
        return build_eag_unet(base=24, gate_mode="none")
    if model_name == "EAG_FullGate_B24":
        return build_eag_unet(base=24, gate_mode="full")
    if model_name == "EAG_ShallowGate_B24":
        return build_eag_unet(base=24, gate_mode="shallow")
    if model_name == "EAG_DeepGate_B24":
        return build_eag_unet(base=24, gate_mode="deep")
    if model_name == "EAG_FullGate_B16":
        return build_eag_unet(base=16, gate_mode="full")
    if model_name == "EAG_FullGate_B32":
        return build_eag_unet(base=32, gate_mode="full")
    raise ValueError(f"Unknown model_name={model_name}")

## 6. Checkpoint-based predictions and genuine segmentation-targeted Grad-CAM

In [ ]:
PARAMS={'U_Net':4080937,'ResUNet':4182649,'Attention_UNet':4130441,
        'EAG_UNet':4130441,'MS_SE_Attention_UNet':5253038}
def to_local(p):
    s=str(p)
    if os.name=='nt' and s.startswith('/mnt/d/'):return 'D:/'+s[len('/mnt/d/'):]
    return s
def image_mask(row):
    im=np.asarray(Image.open(to_local(row['input_path'])).convert('RGB').resize((512,512)))
    gt=(np.asarray(Image.open(to_local(row['reference_mask'])).convert('L').resize(
        (512,512),Image.Resampling.NEAREST))>127)
    return im,gt
def get_feature_layer(model):
    options=[l for l in model.layers if isinstance(l,layers.Conv2D)
             and len(l.output.shape)==4 and l.output.shape[1] is not None
             and 16<=int(l.output.shape[1])<=128]
    assert options
    return options[-1]
def segmentation_gradcam(model,image,threshold):
    feature=get_feature_layer(model)
    probe=tf.keras.Model(model.inputs,[feature.output,model.output])
    with tf.GradientTape() as tape:
        f,y=probe(tf.convert_to_tensor(image[None].astype('float32')/255.),training=False)
        roi=tf.stop_gradient(tf.cast(y>threshold,tf.float32))
        score=tf.reduce_sum(y*roi)/(tf.reduce_sum(roi)+1e-6)
    g=tape.gradient(score,f)
    assert g is not None,'Gradient unavailable'
    w=tf.reduce_mean(g,axis=[1,2],keepdims=True)
    cam=tf.nn.relu(tf.reduce_sum(f*w,axis=-1))[0].numpy()
    cam=cv2.resize(cam,(512,512))
    cam/=np.max(cam)+1e-8
    assert np.isfinite(cam).all()
    return cam,y[0,...,0].numpy(),feature.name
if RUN_GRADCAM:
    rows=selected.to_dict('records');examples=[image_mask(r) for r in rows]
    preds={};cams={};layers_used={}
    for m in MODELS:
        tf.keras.backend.clear_session()
        net=get_model(m)
        assert net.count_params()==PARAMS[m]
        net.load_weights(str(RUNS/m/'seed_42'/'best.weights.h5'))
        thresh=float(raw[(raw.model==m)&(raw.seed==42)].threshold.iloc[0])
        preds[m]=[];cams[m]=[]
        for im,gt in examples:
            heat,prob,name=segmentation_gradcam(net,im,thresh)
            cams[m].append(heat);preds[m].append(prob>=thresh)
        layers_used[m]=name
        print('Checkpoint verified:',m,'gradcam layer:',name,'threshold:',thresh)
        del net;gc.collect();tf.keras.backend.clear_session()
    fig,axs=plt.subplots(3,7,figsize=(17,9))
    for i,(im,gt) in enumerate(examples):
        axs[i,0].imshow(im);axs[i,0].set_title('Fundus input')
        axs[i,1].imshow(gt,cmap='gray',vmin=0,vmax=1);axs[i,1].set_title('Ground truth')
        for j,m in enumerate(MODELS):
            axs[i,j+2].imshow(preds[m][i],cmap='gray',vmin=0,vmax=1)
            axs[i,j+2].set_title(NAMES[m])
        for ax in axs[i]:ax.axis('off')
    fig.suptitle('Figure 7. Actual predicted masks vs ground truth, three preselected FIVES cases')
    fig.tight_layout();save('fig07_prediction_grid.png')
    fig,axs=plt.subplots(3,6,figsize=(15,9))
    for i,(im,gt) in enumerate(examples):
        axs[i,0].imshow(im);axs[i,0].set_title('Fundus input')
        for j,m in enumerate(MODELS):
            axs[i,j+1].imshow(im,alpha=.75)
            axs[i,j+1].imshow(cams[m][i],cmap='inferno',alpha=.40,vmin=0,vmax=1)
            axs[i,j+1].set_title(NAMES[m])
        for ax in axs[i]:ax.axis('off')
    fig.suptitle('Figure 8. Segmentation-targeted Grad-CAM, seed 42 (attribution only)')
    fig.tight_layout();save('fig08_gradcam_grid.png')
    pd.DataFrame([{'model':m,'seed':42,'feature_layer':layers_used[m],
                   'gradient_target':'Mean vessel probability inside predicted foreground'} for m in MODELS]).to_csv(
        TAB/'gradcam_attribution_manifest.csv',index=False)
else:
    print('RUN_GRADCAM False: figures 7-8 were NOT generated')

## 7. Scientific interpretation and limitations
EAG-UNet vs Attention U-Net: difference is small, three-seed paired p not significant, and both implement established additive attention. MS-SE gains are empirical comparisons, not proof of new component design. No claim of clinical biomarker validity. Original FIVES source images, masks, model weights and patient-level metadata are not redistributed in this public repository. The DRIVE transfer set is 20 official TRAINING images, not official test. Grad-CAM is a qualitative model-attribution overlay and does not measure segmentation quality. Three-seed confidence intervals are wide and exploratory, not definitive. Historical ablation and VDRG notebooks are stored separately, not merged into this audit-corrected study.

# Visual results and verified statistical tables

| Model | Parameters | FIVES Dice mean +/- SD | clDice mean +/- SD | DRIVE transfer Dice mean +/- SD |
|---|---:|---:|---:|---:|
| U-Net | 4,080,937 | 0.847105 +/- 0.000890 | 0.870563 +/- 0.000375 | 0.759225 +/- 0.004027 |
| ResUNet | 4,182,649 | 0.848770 +/- 0.004859 | 0.872108 +/- 0.003137 | 0.763081 +/- 0.002029 |
| Attention U-Net | 4,130,441 | 0.849395 +/- 0.003266 | 0.872888 +/- 0.002133 | 0.759897 +/- 0.003266 |
| EAG-UNet | 4,130,441 | 0.849967 +/- 0.003254 | 0.872827 +/- 0.001335 | 0.759794 +/- 0.005744 |
| MS-SE Attention U-Net | 5,253,038 | 0.888042 +/- 0.004624 | 0.893107 +/- 0.003100 | 0.766352 +/- 0.007234 |

Paired comparisons (n=3):

| Comparison (candidate minus reference) | Mean Delta Dice | 95% CI | Paired t p |
|---|---:|---:|---:|
| EAG_UNet vs U_Net | 0.002861 | [-0.003194, 0.008916] | 0.1791 |
| EAG_UNet vs ResUNet | 0.001197 | [-0.003590, 0.005984] | 0.3946 |
| EAG_UNet vs Attention_UNet | 0.000572 | [-0.003198, 0.004342] | 0.5807 |
| EAG_UNet vs MS_SE_Attention_UNet | -0.038075 | [-0.055440, -0.020711] | 0.0110 |
| MS_SE_Attention_UNet vs Attention_UNet | 0.038648 | [0.019651, 0.057644] | 0.0128 |

### FIVES three-seed Dice

![ FIVES three-seed Dice ](figures/fig01_fives_dice.png)

### IoU, clDice, boundary F1 and thin-vessel recall

![ IoU, clDice, boundary F1 and thin-vessel recall ](figures/fig02_four_metrics.png)

### DRIVE training-partition transfer

![ DRIVE training-partition transfer ](figures/fig03_drive_training_transfer.png)

### Seed variability

![ Seed variability ](figures/fig04_seed_variability.png)

### Validation threshold sensitivity

![ Validation threshold sensitivity ](figures/fig05_threshold_sensitivity.png)

### Per-image FIVES Dice

![ Per-image FIVES Dice ](figures/fig06_image_level_boxplots.png)

### Checkpoint-derived predictions

![ Checkpoint-derived predictions ](figures/fig07_prediction_grid.png)

### Segmentation-targeted Grad-CAM

![ Segmentation-targeted Grad-CAM ](figures/fig08_gradcam_grid.png)